# Users Management Agent (MCP) on Google Colab

Runs the whole project inside this notebook:

1. **Mock User Service** on port 8041 (stands in for the Docker image)
2. **MCP server** on port 8005 (5 user tools, 1 resource, 2 prompts)
3. **Agent chat** (OpenAI model + MCP tools) in a cell

**Before you start:** add your OpenAI key in Colab's **Secrets** panel (🔑 icon in the left sidebar) with the
name `OPENAI_API_KEY`, and turn on *Notebook access*. If you skip this, the notebook will ask for the key.

Then use **Runtime → Run all**. The last cell opens a chat box; type `exit` to leave.

## 1. Install dependencies

In [ ]:
# mcp<2: this project uses the MCP SDK v1 API (mcp.server.fastmcp)
!pip install -q "fastmcp==2.10.1" "mcp<2" "openai==1.93.0" requests

## 2. Write the project files
These are the same files as the local project.

In [ ]:
import os
for d in ["mock_user_service", "mcp_server/models", "agent/models"]:
    os.makedirs(os.path.join("/content/ai-general-mcp-basics-main", d), exist_ok=True)
# empty package markers (%%writefile can't write an empty file)
for pkg in ["mcp_server/models", "agent/models"]:
    open(os.path.join("/content/ai-general-mcp-basics-main", pkg, "__init__.py"), "a").close()
print("Folders ready")

In [ ]:
%%writefile /content/ai-general-mcp-basics-main/mock_user_service/server.py
"""
Minimal stand-in for the User Service (khshanovskyi/mockuserservice) for machines without Docker.

- Standard library only; run with:  python mock_user_service/server.py
- Listens on http://localhost:8041 (same port as docker-compose.yml)
- Implements the endpoints used by mcp_server/user_client.py:
    GET    /v1/users/{id}
    GET    /v1/users/search?name=&surname=&email=&gender=
    POST   /v1/users          -> 201
    PUT    /v1/users/{id}     -> 201 (partial update; null fields are ignored)
    DELETE /v1/users/{id}     -> 204
    GET    /health
- Data lives in memory only and resets on restart. No validation beyond unique email.
"""
import json
import re
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from threading import Lock
from urllib.parse import urlparse, parse_qs

HOST, PORT = "127.0.0.1", 8041


def _user(uid, name, surname, email, gender, company, about_me):
    return {
        "id": uid, "name": name, "surname": surname, "email": email,
        "phone": None, "date_of_birth": None, "address": None,
        "gender": gender, "company": company, "salary": None,
        "about_me": about_me, "credit_card": None,
    }


USERS = {u["id"]: u for u in [
    _user(1, "John", "Smith", "john.smith@gmail.com", "male", "Northwind", "Weekend hiker and amateur photographer."),
    _user(2, "Johnny", "Cash", "johnny.cash@acme.com", "male", "Acme", "Plays guitar and restores old radios."),
    _user(3, "Alice", "Johnson", "alice.johnson@gmail.com", "female", "Globex", "Avid reader who loves board games."),
    _user(4, "Maria", "Garcia", "maria.garcia@outlook.com", "female", "Initech", "Marathon runner and home cook."),
    _user(5, "Liam", "Brown", "liam.brown@acme.com", "male", "Acme", "Cyclist, chess player and coffee nerd."),
    _user(6, "Sam", "Taylor", "sam.taylor@yahoo.com", "other", None, "Volunteers at the animal shelter."),
]}
_next_id = max(USERS) + 1
_lock = Lock()


class Handler(BaseHTTPRequestHandler):
    def _send(self, code, obj=None):
        body = b"" if obj is None else json.dumps(obj).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def _json_body(self):
        length = int(self.headers.get("Content-Length") or 0)
        try:
            return json.loads(self.rfile.read(length) or b"{}")
        except json.JSONDecodeError:
            return None

    def _user_id(self, path):
        m = re.fullmatch(r"/v1/users/(\d+)", path)
        return int(m[1]) if m else None

    def do_GET(self):
        url = urlparse(self.path)
        if url.path == "/health":
            return self._send(200, {"status": "ok"})
        if url.path == "/v1/users/search":
            filters = {k: v[0].lower() for k, v in parse_qs(url.query).items()
                       if k in ("name", "surname", "email", "gender")}

            def matches(u):
                for field, value in filters.items():
                    actual = (u.get(field) or "").lower()
                    if (actual != value) if field == "gender" else (value not in actual):
                        return False
                return True

            return self._send(200, [u for u in USERS.values() if matches(u)])
        uid = self._user_id(url.path)
        if uid is not None:
            if uid in USERS:
                return self._send(200, USERS[uid])
            return self._send(404, {"detail": f"User {uid} not found"})
        self._send(404, {"detail": "Not found"})

    def do_POST(self):
        global _next_id
        if urlparse(self.path).path != "/v1/users":
            return self._send(404, {"detail": "Not found"})
        data = self._json_body()
        if not isinstance(data, dict):
            return self._send(422, {"detail": "Body must be a JSON object"})
        missing = [f for f in ("name", "surname", "email", "about_me") if not data.get(f)]
        if missing:
            return self._send(422, {"detail": f"Missing required fields: {', '.join(missing)}"})
        with _lock:
            if any(u["email"].lower() == data["email"].lower() for u in USERS.values()):
                return self._send(409, {"detail": f"Email {data['email']} already exists"})
            uid = _next_id
            _next_id += 1
            USERS[uid] = {**_user(uid, None, None, None, None, None, None), **data, "id": uid}
        self._send(201, USERS[uid])

    def do_PUT(self):
        uid = self._user_id(urlparse(self.path).path)
        if uid is None or uid not in USERS:
            return self._send(404, {"detail": f"User {uid} not found"})
        data = self._json_body()
        if not isinstance(data, dict):
            return self._send(422, {"detail": "Body must be a JSON object"})
        with _lock:
            USERS[uid].update({k: v for k, v in data.items() if v is not None and k != "id"})
        self._send(201, USERS[uid])

    def do_DELETE(self):
        uid = self._user_id(urlparse(self.path).path)
        if uid is None or uid not in USERS:
            return self._send(404, {"detail": f"User {uid} not found"})
        with _lock:
            del USERS[uid]
        self._send(204)

    def log_message(self, fmt, *args):
        print(f"[mock-user-service] {self.command} {self.path} -> {args[1] if len(args) > 1 else ''}")


if __name__ == "__main__":
    print(f"Mock User Service running on http://localhost:{PORT} with {len(USERS)} users (Ctrl+C to stop)")
    try:
        ThreadingHTTPServer((HOST, PORT), Handler).serve_forever()
    except KeyboardInterrupt:
        pass

In [ ]:
%%writefile /content/ai-general-mcp-basics-main/mcp_server/models/user_info.py
from typing import Optional

from pydantic import BaseModel


class Address(BaseModel):
    country: str
    city: str
    street: str
    flat_house: str


class CreditCard(BaseModel):
    num: str
    cvv: str
    exp_date: str


class UserCreate(BaseModel):
    name: str
    surname: str
    email: str
    phone: Optional[str] = None
    date_of_birth: Optional[str] = None
    address: Optional[Address] = None
    gender: Optional[str] = None
    company: Optional[str] = None
    salary: Optional[float] = None
    about_me: str
    credit_card: Optional[CreditCard] = None


class UserUpdate(BaseModel):
    name: Optional[str] = None
    surname: Optional[str] = None
    email: Optional[str] = None
    phone: Optional[str] = None
    date_of_birth: Optional[str] = None
    address: Optional[Address] = None
    gender: Optional[str] = None
    company: Optional[str] = None
    salary: Optional[float] = None
    credit_card: Optional[CreditCard] = None


class UserSearchRequest(BaseModel):
    name: Optional[str] = None
    email: Optional[str] = None
    surname: Optional[str] = None
    gender: Optional[str] = None

In [ ]:
%%writefile /content/ai-general-mcp-basics-main/mcp_server/user_client.py
import os
from typing import Any, Optional

import requests

from models.user_info import UserUpdate, UserCreate

USER_SERVICE_ENDPOINT = os.getenv("USERS_MANAGEMENT_SERVICE_URL", "http://localhost:8041")

class UserClient:

    def __user_to_string(self, user: dict[str, Any]):
        user_str = "```\n"
        for key, value in user.items():
            user_str += f"  {key}: {value}\n"
        user_str += "```\n"

        return user_str

    def __users_to_string(self, users: list[dict[str, Any]]):
        users_str = ""
        for value in users:
            users_str += self.__user_to_string(value)
        users_str += "\n"

        return users_str

    async def get_user(self, user_id: int) -> str:
        headers = {"Content-Type": "application/json"}

        response = requests.get(url=f"{USER_SERVICE_ENDPOINT}/v1/users/{user_id}", headers=headers)

        if response.status_code == 200:
            data = response.json()
            return self.__user_to_string(data)

        raise Exception(f"HTTP {response.status_code}: {response.text}")

    async def search_users(
            self,
            name: Optional[str] = None,
            surname: Optional[str] = None,
            email: Optional[str] = None,
            gender: Optional[str] = None,
    ) -> str:
        headers = {"Content-Type": "application/json"}

        params = {}
        if name:
            params["name"] = name
        if surname:
            params["surname"] = surname
        if email:
            params["email"] = email
        if gender:
            params["gender"] = gender

        response = requests.get(url=USER_SERVICE_ENDPOINT + "/v1/users/search", headers=headers, params=params)

        if response.status_code == 200:
            data = response.json()
            print(f"Get {len(data)} users successfully")
            return self.__users_to_string(data)

        raise Exception(f"HTTP {response.status_code}: {response.text}")

    async def add_user(self, user_create_model: UserCreate) -> str:
        headers = {"Content-Type": "application/json"}

        response = requests.post(
            url=f"{USER_SERVICE_ENDPOINT}/v1/users",
            headers=headers,
            json=user_create_model.model_dump()
        )

        if response.status_code == 201:
            return f"User successfully added: {response.text}"

        raise Exception(f"HTTP {response.status_code}: {response.text}")

    async def update_user(self, user_id: int, user_update_model: UserUpdate) -> str:
        headers = {"Content-Type": "application/json"}

        response = requests.put(
            url=f"{USER_SERVICE_ENDPOINT}/v1/users/{user_id}",
            headers=headers,
            json=user_update_model.model_dump()
        )

        if response.status_code == 201:
            return f"User successfully updated: {response.text}"

        raise Exception(f"HTTP {response.status_code}: {response.text}")

    async def delete_user(self, user_id: int) -> str:
        headers = {"Content-Type": "application/json"}

        response = requests.delete(url=f"{USER_SERVICE_ENDPOINT}/v1/users/{user_id}", headers=headers)

        if response.status_code == 204:
            return "User successfully deleted"

        raise Exception(f"HTTP {response.status_code}: {response.text}")

In [ ]:
%%writefile /content/ai-general-mcp-basics-main/mcp_server/server.py
from pathlib import Path
from typing import Optional

from mcp.server.fastmcp import FastMCP

from models.user_info import UserSearchRequest, UserCreate, UserUpdate
from user_client import UserClient

mcp = FastMCP(
    name="users-management-mcp-server",
    host="0.0.0.0",
    port=8005,
)
user_client = UserClient()


# ==================== TOOLS ====================

@mcp.tool()
async def get_user_by_id(user_id: int) -> str:
    """Get the full profile of a single user by their numeric ID."""
    return await user_client.get_user(user_id)


@mcp.tool()
async def delete_user(user_id: int) -> str:
    """Permanently delete a user by their numeric ID. Only call this when the user has clearly asked for deletion."""
    return await user_client.delete_user(user_id)


@mcp.tool()
async def search_user(
        name: Optional[str] = None,
        surname: Optional[str] = None,
        email: Optional[str] = None,
        gender: Optional[str] = None,
) -> str:
    """Search users. All filters are optional and combined with AND.
    name, surname and email use partial, case-insensitive matching;
    gender is an exact match (male, female, other, prefer_not_to_say)."""
    return await user_client.search_users(name=name, surname=surname, email=email, gender=gender)


@mcp.tool()
async def add_user(user_create_model: UserCreate) -> str:
    """Create a new user. name, surname, email and about_me are required; the email must be unique."""
    return await user_client.add_user(user_create_model)


@mcp.tool()
async def update_user(user_id: int, new_info: UserUpdate) -> str:
    """Update an existing user by ID. Only include the fields that should change."""
    return await user_client.update_user(user_id=user_id, user_update_model=new_info)


# ==================== MCP RESOURCES ====================

@mcp.resource(
    "users-management://flow-diagram",
    mime_type="image/png",
    description="PNG diagram of the User Service's Swagger endpoints and how this MCP server uses them.",
)
async def get_flow_diagram() -> bytes:
    return (Path(__file__).parent / "flow.png").read_bytes()


# ==================== MCP PROMPTS ====================

@mcp.prompt(
    name="user_search_assistant",
    description="Guidance on writing effective user search queries (fields, matching rules, strategies).",
)
async def user_search_assistant() -> str:
    return """
You are helping users search through a dynamic user database. The database contains
realistic synthetic user profiles with the following searchable fields:

## Available Search Parameters
- **name**: First name (partial matching, case-insensitive)
- **surname**: Last name (partial matching, case-insensitive)
- **email**: Email address (partial matching, case-insensitive)
- **gender**: Exact match (male, female, other, prefer_not_to_say)

## Search Strategy Guidance

### For Name Searches
- Use partial names: "john" finds John, Johnny, Johnson, etc.
- Try common variations: "mike" vs "michael", "liz" vs "elizabeth"
- Consider cultural name variations

### For Email Searches
- Search by domain: "gmail" for all Gmail users
- Search by name patterns: "john" for emails containing john
- Use company names to find business emails

### For Demographic Analysis
- Combine gender with other criteria for targeted searches
- Use broad searches first, then narrow down

### Effective Search Combinations
- Name + Gender: Find specific demographic segments
- Email domain + Surname: Find business contacts
- Partial names: Cast wider nets for common names

## Example Search Patterns
```
"Find all Johns" → name="john"
"Gmail users named Smith" → email="gmail" + surname="smith"
"Female users with company emails" → gender="female" + email="company"
"Users with Johnson surname" → surname="johnson"
```

## Tips for Better Results
1. Start broad, then narrow down
2. Try variations of names (John vs Johnny)
3. Use partial matches creatively
4. Combine multiple criteria for precision
5. Remember searches are case-insensitive

When helping users search, suggest multiple search strategies and explain
why certain approaches might be more effective for their goals.
"""


@mcp.prompt(
    name="user_profile_creation_guide",
    description="Guidelines for creating realistic, consistent user profiles (fields, formats, biography).",
)
async def user_profile_creation_guide() -> str:
    return """
You are helping create realistic user profiles for the system. Follow these guidelines
to ensure data consistency and realism.

## Required Fields
- **name**: 2-50 characters, letters only, culturally appropriate
- **surname**: 2-50 characters, letters only
- **email**: Valid format, must be unique in system
- **about_me**: Rich, realistic biography (see guidelines below)

## Optional Fields Best Practices
- **phone**: Use E.164 format (+1234567890) when possible
- **date_of_birth**: YYYY-MM-DD format, realistic ages (18-80)
- **gender**: Use standard values (male, female, other, prefer_not_to_say)
- **company**: Real-sounding company names
- **salary**: $30,000-$200,000 range for employed individuals

## Address Guidelines
Provide complete, realistic addresses:
- **country**: Full country names
- **city**: Actual city names
- **street**: Realistic street addresses
- **flat_house**: Apartment/unit format (Apt 123, Unit 5B, Suite 200)

## Credit Card Guidelines
Generate realistic but non-functional card data:
- **num**: 16 digits formatted as XXXX-XXXX-XXXX-XXXX
- **cvv**: 3 digits (000-999)
- **exp_date**: MM/YYYY format, future dates only

## Biography Creation ("about_me")
Create engaging, realistic biographies that include:

### Personality Elements
- 1-3 personality traits (curious, adventurous, analytical, etc.)
- Authentic voice and writing style
- Cultural and demographic appropriateness

### Interests & Hobbies
- 2-4 specific hobbies or activities
- 1-3 broader interests or passion areas
- 1-2 life goals or aspirations

### Biography Templates
Use varied narrative structures:
- "I'm a [trait] person who loves [hobbies]..."
- "When I'm not working, you can find me [activity]..."
- "Life is all about balance for me. I enjoy [interests]..."
- "As someone who's [trait], I find great joy in [hobby]..."

## Data Validation Reminders
- Email uniqueness is enforced (check existing users)
- Phone numbers should follow consistent formatting
- Date formats must be exact (YYYY-MM-DD)
- Credit card expiration dates must be in the future
- Salary values should be realistic for the demographic

## Cultural Sensitivity
- Match names to appropriate cultural backgrounds
- Consider regional variations in address formats
- Use realistic company names for the user's location
- Ensure hobbies and interests are culturally appropriate

When creating profiles, aim for diversity in:
- Geographic representation
- Age distribution
- Interest variety
- Socioeconomic backgrounds
- Cultural backgrounds
"""


if __name__ == "__main__":
    mcp.run(transport="streamable-http")

In [ ]:
%%writefile /content/ai-general-mcp-basics-main/agent/models/message.py
from enum import StrEnum
from typing import Any
from pydantic import BaseModel


class Role(StrEnum):
    SYSTEM = "system"
    USER = "user"
    AI = "assistant"
    TOOL = "tool"


class Message(BaseModel):
    role: Role
    content: str | None = None
    tool_call_id: str | None = None
    name: str | None = None
    tool_calls: list[dict[str, Any]] | None = None

    def to_dict(self) -> dict[str, Any]:
        result = {"role": str(self.role.value)}
        if self.content:
            result["content"] = self.content
        if self.name:
            result["name"] = self.name
        if self.tool_call_id:
            result["tool_call_id"] = self.tool_call_id
        if self.tool_calls:
            result["tool_calls"] = self.tool_calls
        return result

In [ ]:
%%writefile /content/ai-general-mcp-basics-main/agent/prompts.py
SYSTEM_PROMPT = """
You are the Users Management Agent. You help operators manage records in the User Management Service
using the tools provided by the users-management MCP server.

## What you can do
- Look up a user by ID (get_user_by_id)
- Search users by name, surname, email or gender (search_user)
- Create users (add_user)
- Update users (update_user)
- Delete users (delete_user)

## How to work
- Use the tools for every fact about users. Never invent user data or IDs.
- If a request is ambiguous (for example, several users match a name), search first, show the candidates,
  and ask which one is meant before changing anything.
- Before deleting a user, state which user (ID and name) will be deleted and ask for confirmation.
  Only call delete_user after the operator confirms.
- For create and update, send only the fields you were given or that are clearly implied. Ask for any
  missing required fields (name, surname, email, about_me) instead of guessing them.
- If a tool returns an error, explain it plainly and suggest what to try next.

## Limits
- You have no web access. You cannot look people up on the internet; work only with the User Service.
- Stay within user management. Politely decline unrelated requests.
- Do not show full credit card numbers or CVVs unless the operator explicitly asks for them.

## Style
- Be concise and professional.
- Present users as short, structured summaries (ID, name, email, and the fields that matter for the request).
- After any create, update or delete, confirm what changed.
"""

In [ ]:
%%writefile /content/ai-general-mcp-basics-main/agent/mcp_client.py
from typing import Optional, Any

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from mcp.types import CallToolResult, TextContent, GetPromptResult, ReadResourceResult, Resource, TextResourceContents, BlobResourceContents, Prompt
from pydantic import AnyUrl


class MCPClient:
    """Handles MCP server connection and tool execution"""

    def __init__(self, mcp_server_url: str) -> None:
        self.mcp_server_url = mcp_server_url
        self.session: Optional[ClientSession] = None
        self._streams_context = None
        self._session_context = None

    async def __aenter__(self):
        self._streams_context = streamablehttp_client(self.mcp_server_url)
        read_stream, write_stream, _ = await self._streams_context.__aenter__()

        self._session_context = ClientSession(read_stream, write_stream)
        self.session = await self._session_context.__aenter__()

        init_result = await self.session.initialize()
        print(f"Connected to MCP server: {init_result.model_dump_json(indent=2)}\n")
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        if self.session and self._session_context:
            await self._session_context.__aexit__(exc_type, exc_val, exc_tb)
        if self._streams_context:
            await self._streams_context.__aexit__(exc_type, exc_val, exc_tb)

    async def get_tools(self) -> list[dict[str, Any]]:
        """Get available tools from MCP server"""
        if not self.session:
            raise RuntimeError("MCP client not connected. Call connect() first.")
        tools = await self.session.list_tools()
        return [
            {
                "type": "function",
                "function": {
                    "name": tool.name,
                    "description": tool.description,
                    "parameters": tool.inputSchema,
                },
            }
            for tool in tools.tools
        ]

    async def call_tool(self, tool_name: str, tool_args: dict[str, Any]) -> Any:
        """Call a specific tool on the MCP server"""
        if not self.session:
            raise RuntimeError("MCP client not connected. Call connect() first.")

        tool_result: CallToolResult = await self.session.call_tool(tool_name, tool_args)
        content = tool_result.content[0]
        print(f"    ⚙️: {content}\n")
        if isinstance(content, TextContent):
            return content.text
        return content

    async def get_resources(self) -> list[Resource]:
        """Get available resources from MCP server"""
        if not self.session:
            raise RuntimeError("MCP client not connected.")
        try:
            result = await self.session.list_resources()
            return result.resources
        except Exception as e:
            print(f"Could not list resources: {e}")
            return []

    async def get_resource(self, uri: AnyUrl) -> str:
        """Get specific resource content"""
        if not self.session:
            raise RuntimeError("MCP client not connected.")

        result: ReadResourceResult = await self.session.read_resource(uri)
        content = result.contents[0]
        if isinstance(content, TextResourceContents):
            return content.text
        if isinstance(content, BlobResourceContents):
            return content.blob
        return str(content)

    async def get_prompts(self) -> list[Prompt]:
        """Get available prompts from MCP server"""
        if not self.session:
            raise RuntimeError("MCP client not connected.")
        try:
            result = await self.session.list_prompts()
            return result.prompts
        except Exception as e:
            print(f"Could not list prompts: {e}")
            return []

    async def get_prompt(self, name: str) -> str:
        """Get specific prompt content"""
        if not self.session:
            raise RuntimeError("MCP client not connected.")
        result: GetPromptResult = await self.session.get_prompt(name)
        combined_content = ""
        for message in result.messages:
            content = getattr(message, "content", None)
            if isinstance(content, TextContent):
                combined_content += content.text + "\n"
            elif isinstance(content, str):
                combined_content += content + "\n"
        return combined_content

In [ ]:
%%writefile /content/ai-general-mcp-basics-main/agent/openai_client.py
import json
from collections import defaultdict
from typing import Any

from openai import AsyncOpenAI

from agent.models.message import Message, Role
from agent.mcp_client import MCPClient


class OpenAIClient:
    """Handles AI model interactions and integrates with MCP client"""

    def __init__(self, api_key: str, model: str, tools: list[dict[str, Any]], mcp_client: MCPClient):
        self.model=model
        self.tools = tools
        self.mcp_client = mcp_client
        self.openai = AsyncOpenAI(api_key=api_key)

    def _collect_tool_calls(self, tool_deltas):
        """Convert streaming tool call deltas to complete tool calls"""
        tool_dict = defaultdict(lambda: {"id": None, "function": {"arguments": "", "name": None}, "type": None})

        for delta in tool_deltas:
            idx = delta.index
            if delta.id: tool_dict[idx]["id"] = delta.id
            if delta.function.name: tool_dict[idx]["function"]["name"] = delta.function.name
            if delta.function.arguments: tool_dict[idx]["function"]["arguments"] += delta.function.arguments
            if delta.type: tool_dict[idx]["type"] = delta.type

        return list(tool_dict.values())

    async def _stream_response(self, messages: list[Message]) -> Message:
        """Stream OpenAI response and handle tool calls"""
        stream = await self.openai.chat.completions.create(
            **{
                "model": self.model,
                "messages": [msg.to_dict() for msg in messages],
                "tools": self.tools,
                "temperature": 0.0,
                "stream": True
            }
        )

        content = ""
        tool_deltas = []

        print("🤖: ", end="", flush=True)

        async for chunk in stream:
            delta = chunk.choices[0].delta

            # Stream content
            if delta.content:
                print(delta.content, end="", flush=True)
                content += delta.content

            if delta.tool_calls:
                tool_deltas.extend(delta.tool_calls)

        print()
        return Message(
            role=Role.AI,
            content=content,
            tool_calls=self._collect_tool_calls(tool_deltas) if tool_deltas else []
        )

    async def get_completion(self, messages: list[Message]) -> Message:
        """Process user query with streaming and tool calling"""
        ai_message: Message = await self._stream_response(messages)

        # Check if any tool calls are present and perform them
        if ai_message.tool_calls:
            messages.append(ai_message)
            await self._call_tools(ai_message, messages)
            # recursively calling agent with tool messages
            return await self.get_completion(messages)

        return ai_message

    async def _call_tools(self, ai_message: Message, messages: list[Message]):
        """Execute tool calls using MCP client"""
        for tool_call in ai_message.tool_calls:
            tool_name = tool_call["function"]["name"]
            try:
                tool_args = json.loads(tool_call["function"]["arguments"] or "{}")
                result = await self.mcp_client.call_tool(tool_name, tool_args)
                content = result if isinstance(result, str) else str(result)
            except Exception as e:
                content = f"Error while calling tool '{tool_name}': {e}"
                print(f"    ❌: {content}\n")

            messages.append(
                Message(
                    role=Role.TOOL,
                    name=tool_name,
                    tool_call_id=tool_call["id"],
                    content=content,
                )
            )

In [ ]:
%%writefile /content/ai-general-mcp-basics-main/agent/app.py
import asyncio
import json
import os
import sys

from mcp import Resource
from mcp.types import Prompt

from agent.mcp_client import MCPClient
from agent.openai_client import OpenAIClient
from agent.models.message import Message, Role
from agent.prompts import SYSTEM_PROMPT


# https://remote.mcpservers.org/fetch/mcp
# Pay attention that `fetch` doesn't have resources and prompts

MCP_SERVER_URL = os.getenv("MCP_SERVER_URL", "http://localhost:8005/mcp")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o")


async def main():
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise SystemExit("OPENAI_API_KEY is not set. Set it in your environment and run again.")

    async with MCPClient(mcp_server_url=MCP_SERVER_URL) as mcp_client:
        resources: list[Resource] = await mcp_client.get_resources()
        print("Available resources:")
        for resource in resources:
            print(f"  - {resource.uri} ({resource.mimeType}): {resource.description}")
        print()

        tools = await mcp_client.get_tools()
        print("Available tools:")
        for tool in tools:
            print(json.dumps(tool, indent=2))
        print()

        openai_client = OpenAIClient(
            api_key=api_key,
            model=OPENAI_MODEL,
            tools=tools,
            mcp_client=mcp_client,
        )

        messages: list[Message] = [Message(role=Role.SYSTEM, content=SYSTEM_PROMPT)]

        prompts: list[Prompt] = await mcp_client.get_prompts()
        for prompt in prompts:
            prompt_text = await mcp_client.get_prompt(prompt.name)
            messages.append(Message(role=Role.USER, content=prompt_text))
            print(f"Loaded MCP prompt: {prompt.name}")
        print()

        print("Users Management Agent is ready. Type 'exit' or 'quit' to leave.\n")
        while True:
            try:
                user_input = input("👤: ").lstrip("﻿").strip()
            except (EOFError, KeyboardInterrupt):
                print()
                break

            if not user_input:
                continue
            if user_input.lower() in ("exit", "quit"):
                break

            history_len = len(messages)
            messages.append(Message(role=Role.USER, content=user_input))
            try:
                ai_message = await openai_client.get_completion(messages)
                messages.append(ai_message)
            except Exception as e:
                # Roll back this turn so a failed call doesn't leave half a conversation in history.
                del messages[history_len:]
                print(f"\n❌ Request failed: {e}")
            print()


if __name__ == "__main__":
    # The chat prints emoji; Windows consoles/pipes may default to cp1252, which can't encode them.
    sys.stdout.reconfigure(encoding="utf-8")
    asyncio.run(main())

In [ ]:
# flow.png is served by the MCP server as the `users-management://flow-diagram` resource
import base64
FLOW_PNG_B64 = "..."
with open("/content/ai-general-mcp-basics-main/mcp_server/flow.png", "wb") as f:
    f.write(base64.b64decode(FLOW_PNG_B64))
print("flow.png written")

## 3. Start the Mock User Service and the MCP server in the background
Logs go to `mock.log` and `mcp.log` in the project folder.

In [ ]:
import socket, subprocess, sys, time

BASE = "/content/ai-general-mcp-basics-main"

def port_open(port):
    with socket.socket() as s:
        s.settimeout(0.5)
        return s.connect_ex(("127.0.0.1", port)) == 0

def start(name, args, cwd, port, log):
    if port_open(port):
        print(f"{name} already running on port {port}")
        return None
    proc = subprocess.Popen([sys.executable, "-u", *args], cwd=cwd,
                            stdout=open(log, "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if port_open(port):
            print(f"{name} running on port {port} (pid {proc.pid})")
            return proc
        if proc.poll() is not None:
            break
        time.sleep(0.5)
    print(f"{name} failed to start. Log:")
    print(open(log).read())
    raise RuntimeError(f"{name} did not start")

mock_proc = start("Mock User Service", ["mock_user_service/server.py"], BASE, 8041, f"{BASE}/mock.log")
mcp_proc = start("MCP server", ["server.py"], f"{BASE}/mcp_server", 8005, f"{BASE}/mcp.log")

## 4. Optional: check the MCP tools without OpenAI
Connects to the MCP server and calls `search_user` directly.

In [ ]:
import sys
if "/content/ai-general-mcp-basics-main" not in sys.path:
    sys.path.insert(0, "/content/ai-general-mcp-basics-main")

from agent.mcp_client import MCPClient

async with MCPClient("http://localhost:8005/mcp") as client:
    tools = await client.get_tools()
    print("Tools:", [t["function"]["name"] for t in tools])
    print("Resources:", [str(r.uri) for r in await client.get_resources()])
    print("Prompts:", [p.name for p in await client.get_prompts()])
    print(await client.call_tool("search_user", {"name": "john"}))

## 5. OpenAI key
Reads `OPENAI_API_KEY` from Colab Secrets, or asks for it (the input is hidden).

In [ ]:
import os
from getpass import getpass

key = None
try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
except Exception as e:
    print(f"Could not read Colab secret ({type(e).__name__}); asking instead.")
if not key:
    key = getpass("OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = key
os.environ.setdefault("OPENAI_MODEL", "gpt-4o")  # change to any chat model your key can use
print("Key set. Model:", os.environ["OPENAI_MODEL"])

## 6. Chat with the agent
Try: `find users named john`, `add a user Emma Wilson, emma@example.com, loves sailing`, `delete user 6`.
Type `exit` to end the chat (then you can run this cell again to start a fresh one).

In [ ]:
import os, sys
os.chdir("/content/ai-general-mcp-basics-main")
if "/content/ai-general-mcp-basics-main" not in sys.path:
    sys.path.insert(0, "/content/ai-general-mcp-basics-main")

from agent.app import main
await main()

## 7. Stop the background servers

In [ ]:
for proc in (mcp_proc, mock_proc):
    if proc and proc.poll() is None:
        proc.terminate()
print("Stopped")